<a href="https://www.kaggle.com/code/izzarsulynashrudin/pcqm4mv2?scriptVersionId=308970694" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Baseline PCQM4Mv2

Notebook ini menampilkan baseline prediksi `radiusGyration` pada data training PCQM4Mv2 dengan membandingkan model tabular dan graph neural network.

Model yang dipakai mengikuti implementasi di `main.py`, yaitu `ridgeRegression`, `randomForest`, `extraTrees`, `gradientBoosting`, `histGradientBoosting`, `xgboost` bila tersedia, dan `ginGraphNetwork`.


## Import dan Setup

Bagian ini memuat library, pengaturan awal, serta helper dari `main.py` agar notebook dan script utama memakai logika model yang konsisten.


In [ ]:
import json
import math
import os
import pickle
import time
import warnings
from collections import Counter
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import explained_variance_score, mean_absolute_error, mean_squared_error, median_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from main import DEFAULT_FEATURE_COLUMNS, build_model_map, sync_notebook_web_assets, train_gin_regressor

try:
    from xgboost import XGBRegressor
    xgboostAvailable = True
except ImportError:
    XGBRegressor = None
    xgboostAvailable = False

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
sns.set_theme(style='whitegrid', palette='crest')
plt.rcParams['figure.figsize'] = (10, 5)

print('Library utama berhasil dimuat.')
if xgboostAvailable:
    print('XGBoost tersedia dan siap dibandingkan.')


## Ringkasan Data

Bagian ini menampilkan konfigurasi run aktif seperti jumlah molekul, target `radiusGyration`, pembagian data, dan parameter awal yang dipakai oleh seluruh model.


In [ ]:
if True:
        datasetDir = Path('dataset')
        sdfPath = datasetDir / 'pcqm4m-v2-train.sdf'
        outputDir = Path('output')
        notebookPath = Path('pcqm4m.ipynb')
        outputDir.mkdir(parents=True, exist_ok=True)

        maxTrainingMoleculeCount = 3_378_606
        viewerMoleculeCount = 200
        overviewPreviewCount = 3
        edaMoleculeSampleCount = 2_000
        sampleStride = 1
        reportInterval = 5_000
        randomState = 42
        targetColumnName = 'radiusGyration'
        localValidFraction = 0.10
        localTestFraction = 0.10
        reuseCachedFeatures = True

        viewerExportConfig = {
            'enabled': True,
            'outputDir': 'output/data',
            'chunkSize': 2_000,
            'stride': sampleStride,
            'maxMolecules': viewerMoleculeCount,
            'indexMode': 'sequential',
            'layoutMode': 'clustered-3d',
            'moleculesPerRow': 100,
            'rowsPerLayer': 25,
            'cellGap': 8.0,
            'layerGap': 18.0,
            'clusterCount': 1,
            'clusterRadius': 0.0,
            'clusterSpread': 380.0,
            'randomSeed': 1337,
            'depthScale': 1.0,
            'roundDigits': 3,
            'dropHydrogens': False,
            'logEvery': reportInterval,
        }

        showcaseExportConfig = {
            'outputName': 'notebookShowcase.json',
            'webManifestPath': 'output/notebookWebManifest.json',
        }

        modelFeatureColumns = DEFAULT_FEATURE_COLUMNS.copy()

        def iterSdfBlocks(sdfFilePath, maxMoleculeCount=None, sampleStride=1):
            blockLines = []
            seenMoleculeCount = 0
            yieldedCount = 0

            with sdfFilePath.open('r', encoding='utf-8', errors='ignore') as handle:
                for rawLine in handle:
                    textLine = rawLine.rstrip('\n')
                    if textLine == '$$$$':
                        if blockLines:
                            seenMoleculeCount += 1
                            if (seenMoleculeCount - 1) % sampleStride == 0:
                                yield blockLines
                                yieldedCount += 1
                                if maxMoleculeCount is not None and yieldedCount >= maxMoleculeCount:
                                    return
                        blockLines = []
                    else:
                        blockLines.append(textLine)

                if blockLines and (maxMoleculeCount is None or yieldedCount < maxMoleculeCount):
                    yield blockLines

        def parseCountsLine(countsLine):
            try:
                atomCount = int(countsLine[0:3].strip())
                bondCount = int(countsLine[3:6].strip())
                return atomCount, bondCount
            except ValueError:
                parts = countsLine.split()
                if len(parts) < 2:
                    return None
                try:
                    atomCount = int(parts[0])
                    bondCount = int(parts[1])
                    return atomCount, bondCount
                except ValueError:
                    return None

        if not sdfPath.exists():
            raise FileNotFoundError(f'File SDF tidak ditemukan di: {sdfPath}')

        datasetOverviewFrame = pd.DataFrame(
            {
                'parameter': [
                    'datasetDir',
                    'sdfPath',
                    'ukuranFileGB',
                    'maxTrainingMoleculeCount',
                    'viewerMoleculeCount',
                    'sampleStride',
                    'targetColumnName',
                    'splitData',
                ],
                'nilai': [
                    str(datasetDir),
                    str(sdfPath),
                    round(sdfPath.stat().st_size / (1024 ** 3), 2),
                    maxTrainingMoleculeCount,
                    viewerMoleculeCount,
                    sampleStride,
                    targetColumnName,
                    f"train={1.0 - localValidFraction - localTestFraction:.0%}, valid={localValidFraction:.0%}, test={localTestFraction:.0%}",
                ],
            }
        )

        previewRows = []
        for previewIndex, blockLines in enumerate(
            iterSdfBlocks(sdfPath, maxMoleculeCount=overviewPreviewCount, sampleStride=sampleStride),
            start=1,
        ):
            countsTuple = parseCountsLine(blockLines[3]) if len(blockLines) > 3 else None
            atomCount, bondCount = countsTuple if countsTuple is not None else (np.nan, np.nan)
            previewRows.append(
                {
                    'previewIndex': previewIndex,
                    'judulMolekul': blockLines[0].strip() if blockLines and blockLines[0].strip() else f'molecule{previewIndex}',
                    'atomCount': atomCount,
                    'bondCount': bondCount,
                }
            )

        print('Ringkasan konfigurasi:')
        display(datasetOverviewFrame)

        print('Cuplikan molekul awal dari file SDF:')
        display(pd.DataFrame(previewRows))


## Eksplorasi Data

Bagian ini melihat pola dasar struktur molekul seperti jumlah atom dan jumlah ikatan, karena statistik ini nantinya menjadi bagian dari fitur tabular dan juga membantu membaca kompleksitas graf untuk GIN.


In [ ]:
edaRows = []

for sampledIndex, blockLines in enumerate(
    iterSdfBlocks(
        sdfPath,
        maxMoleculeCount=min(edaMoleculeSampleCount, maxTrainingMoleculeCount),
        sampleStride=sampleStride,
    ),
    start=1,
):
    countsTuple = parseCountsLine(blockLines[3]) if len(blockLines) > 3 else None
    atomCount, bondCount = countsTuple if countsTuple is not None else (np.nan, np.nan)
    edaRows.append(
        {
            'sampleIndex': sampledIndex,
            'judulMolekul': blockLines[0].strip() if blockLines and blockLines[0].strip() else f'molecule{sampledIndex}',
            'atomCount': atomCount,
            'bondCount': bondCount,
        }
    )

edaFrame = pd.DataFrame(edaRows)
if edaFrame.empty:
    raise RuntimeError('Eksplorasi data gagal karena tidak ada blok SDF yang berhasil dibaca.')

edaSummaryFrame = pd.DataFrame(
    {
        'metrik': ['jumlahSampelEDA', 'atomCountRataRata', 'bondCountRataRata', 'atomCountMaksimum', 'bondCountMaksimum'],
        'nilai': [
            int(len(edaFrame)),
            float(edaFrame['atomCount'].mean()),
            float(edaFrame['bondCount'].mean()),
            float(edaFrame['atomCount'].max()),
            float(edaFrame['bondCount'].max()),
        ],
    }
)

print('Ringkasan eksplorasi data:')
display(edaSummaryFrame)
display(edaFrame.head(10))

fig, axes = plt.subplots(1, 3, figsize=(19, 5))
sns.histplot(edaFrame['atomCount'], bins=30, kde=True, ax=axes[0], color='#0f766e')
axes[0].set_title('Distribusi Jumlah Atom')
axes[0].set_xlabel('atomCount')

sns.histplot(edaFrame['bondCount'], bins=30, kde=True, ax=axes[1], color='#1d4ed8')
axes[1].set_title('Distribusi Jumlah Ikatan')
axes[1].set_xlabel('bondCount')

sns.scatterplot(data=edaFrame, x='atomCount', y='bondCount', alpha=0.35, ax=axes[2], color='#0f172a', edgecolor=None)
axes[2].set_title('Kepadatan Struktur Awal')
axes[2].set_xlabel('atomCount')
axes[2].set_ylabel('bondCount')

plt.tight_layout()
plt.show()


## Prapemrosesan Data

Bagian ini mengubah setiap molekul menjadi fitur numerik yang konsisten dengan daftar `DEFAULT_FEATURE_COLUMNS` di `main.py`, lalu menyiapkan `radiusGyration` sebagai target prediksi.


In [ ]:
elementNumberMap = {
    'H': 1,
    'C': 6,
    'N': 7,
    'O': 8,
    'F': 9,
    'P': 15,
    'S': 16,
    'Cl': 17,
    'Br': 35,
    'I': 53,
}

elementMassMap = {
    'H': 1.008,
    'C': 12.011,
    'N': 14.007,
    'O': 15.999,
    'F': 18.998,
    'P': 30.974,
    'S': 32.06,
    'Cl': 35.45,
    'Br': 79.904,
    'I': 126.904,
}

def parseAtomLine(atomLine):
    try:
        xValue = float(atomLine[0:10].strip())
        yValue = float(atomLine[10:20].strip())
        zValue = float(atomLine[20:30].strip())
        atomSymbol = atomLine[31:34].strip() or 'X'
        return xValue, yValue, zValue, atomSymbol
    except ValueError:
        parts = atomLine.split()
        if len(parts) < 4:
            return None
        try:
            xValue = float(parts[0])
            yValue = float(parts[1])
            zValue = float(parts[2])
        except ValueError:
            return None
        atomSymbol = parts[3]
        return xValue, yValue, zValue, atomSymbol

def parseBondLine(bondLine):
    try:
        beginIndex = int(bondLine[0:3].strip()) - 1
        endIndex = int(bondLine[3:6].strip()) - 1
        bondOrder = int(bondLine[6:9].strip())
        return beginIndex, endIndex, bondOrder
    except ValueError:
        parts = bondLine.split()
        if len(parts) < 3:
            return None
        try:
            beginIndex = int(parts[0]) - 1
            endIndex = int(parts[1]) - 1
            bondOrder = int(parts[2])
            return beginIndex, endIndex, bondOrder
        except ValueError:
            return None

def countConnectedComponents(atomCount, bondPairs):
    if atomCount == 0:
        return 0

    adjacencyList = [[] for _ in range(atomCount)]
    for beginIndex, endIndex in bondPairs:
        adjacencyList[beginIndex].append(endIndex)
        adjacencyList[endIndex].append(beginIndex)

    visitedFlags = [False] * atomCount
    componentCount = 0

    for startIndex in range(atomCount):
        if visitedFlags[startIndex]:
            continue
        componentCount += 1
        stackList = [startIndex]
        visitedFlags[startIndex] = True
        while stackList:
            currentIndex = stackList.pop()
            for neighborIndex in adjacencyList[currentIndex]:
                if not visitedFlags[neighborIndex]:
                    visitedFlags[neighborIndex] = True
                    stackList.append(neighborIndex)

    return componentCount

def computeRadiusGyration(xCoords, yCoords, zCoords):
    coordinateMatrix = np.column_stack([xCoords, yCoords, zCoords]).astype(float)
    centroidVector = coordinateMatrix.mean(axis=0)
    squaredDistances = np.sum((coordinateMatrix - centroidVector) ** 2, axis=1)
    return float(np.sqrt(squaredDistances.mean()))

def computeBondLengthStats(xCoords, yCoords, zCoords, bondPairs):
    if not bondPairs:
        return 0.0, 0.0

    coordinateMatrix = np.column_stack([xCoords, yCoords, zCoords]).astype(float)
    bondLengths = []
    for beginIndex, endIndex in bondPairs:
        lengthValue = float(np.linalg.norm(coordinateMatrix[beginIndex] - coordinateMatrix[endIndex]))
        bondLengths.append(lengthValue)

    return float(np.mean(bondLengths)), float(np.std(bondLengths))

def extractMoleculeFeatures(blockLines, moleculeIndex):
    if len(blockLines) < 5:
        return None

    countsTuple = parseCountsLine(blockLines[3])
    if countsTuple is None:
        return None

    atomCount, bondCount = countsTuple
    atomStartIndex = 4
    atomEndIndex = atomStartIndex + atomCount
    bondEndIndex = atomEndIndex + bondCount

    if len(blockLines) < bondEndIndex:
        return None

    atomSymbols = []
    xCoords = []
    yCoords = []
    zCoords = []

    for atomLine in blockLines[atomStartIndex:atomEndIndex]:
        atomTuple = parseAtomLine(atomLine)
        if atomTuple is None:
            return None
        xValue, yValue, zValue, atomSymbol = atomTuple
        xCoords.append(xValue)
        yCoords.append(yValue)
        zCoords.append(zValue)
        atomSymbols.append(atomSymbol)

    bondPairs = []
    bondOrders = []
    degreeValues = [0] * atomCount

    for bondLine in blockLines[atomEndIndex:bondEndIndex]:
        bondTuple = parseBondLine(bondLine)
        if bondTuple is None:
            continue
        beginIndex, endIndex, bondOrder = bondTuple
        if 0 <= beginIndex < atomCount and 0 <= endIndex < atomCount:
            bondPairs.append((beginIndex, endIndex))
            bondOrders.append(bondOrder)
            degreeValues[beginIndex] += 1
            degreeValues[endIndex] += 1

    if not atomSymbols:
        return None

    elementCounter = Counter(atomSymbols)
    atomicNumbers = np.array([elementNumberMap.get(symbol, 0) for symbol in atomSymbols], dtype=float)
    atomicMasses = np.array([elementMassMap.get(symbol, 0.0) for symbol in atomSymbols], dtype=float)
    degreeArray = np.array(degreeValues, dtype=float)

    heavyAtomCount = int(sum(symbol != 'H' for symbol in atomSymbols))
    hydrogenCount = int(atomCount - heavyAtomCount)
    heteroAtomCount = int(sum(symbol not in {'C', 'H'} for symbol in atomSymbols))
    componentCount = countConnectedComponents(atomCount, bondPairs)
    ringIndex = max(len(bondPairs) - atomCount + componentCount, 0)
    meanBondLength, stdBondLength = computeBondLengthStats(xCoords, yCoords, zCoords, bondPairs)
    radiusGyration = computeRadiusGyration(xCoords, yCoords, zCoords)

    return {
        'moleculeIndex': moleculeIndex,
        'moleculeTitle': blockLines[0].strip() if blockLines[0].strip() else f'molecule{moleculeIndex}',
        'atomCount': atomCount,
        'heavyAtomCount': heavyAtomCount,
        'hydrogenCount': hydrogenCount,
        'bondCount': len(bondPairs),
        'singleBondCount': int(sum(order == 1 for order in bondOrders)),
        'doubleBondCount': int(sum(order == 2 for order in bondOrders)),
        'tripleBondCount': int(sum(order == 3 for order in bondOrders)),
        'aromaticBondCount': int(sum(order == 4 for order in bondOrders)),
        'uniqueElementCount': int(len(elementCounter)),
        'heteroAtomCount': heteroAtomCount,
        'heteroAtomFraction': float(heteroAtomCount / atomCount),
        'totalAtomicNumber': float(atomicNumbers.sum()),
        'meanAtomicNumber': float(atomicNumbers.mean()),
        'totalAtomicMass': float(atomicMasses.sum()),
        'meanAtomicMass': float(atomicMasses.mean()),
        'degreeMean': float(degreeArray.mean()),
        'degreeStd': float(degreeArray.std()),
        'degreeMax': float(degreeArray.max()),
        'degreeMin': float(degreeArray.min()),
        'terminalAtomFraction': float(np.mean(degreeArray == 1)),
        'branchAtomFraction': float(np.mean(degreeArray >= 3)),
        'bondToAtomRatio': float(len(bondPairs) / atomCount),
        'ringIndex': float(ringIndex),
        'meanBondLength': meanBondLength,
        'stdBondLength': stdBondLength,
        'elementCountC': int(elementCounter.get('C', 0)),
        'elementCountH': int(elementCounter.get('H', 0)),
        'elementCountN': int(elementCounter.get('N', 0)),
        'elementCountO': int(elementCounter.get('O', 0)),
        'elementCountF': int(elementCounter.get('F', 0)),
        'elementCountP': int(elementCounter.get('P', 0)),
        'elementCountS': int(elementCounter.get('S', 0)),
        'elementCountCl': int(elementCounter.get('Cl', 0)),
        'elementCountBr': int(elementCounter.get('Br', 0)),
        'elementCountI': int(elementCounter.get('I', 0)),
        'radiusGyration': radiusGyration,
    }

featureCsvPath = outputDir / 'fiturRegresiSdf.csv'
featureMetadataPath = outputDir / 'fiturRegresiSdf.metadata.json'
cacheIsReusable = False
cachedFeatureMetadata = {}

if reuseCachedFeatures and featureCsvPath.exists() and featureMetadataPath.exists():
    with featureMetadataPath.open('r', encoding='utf-8') as handle:
        cachedFeatureMetadata = json.load(handle)

    cacheIsReusable = (
        cachedFeatureMetadata.get('sdfPath') == str(sdfPath)
        and int(cachedFeatureMetadata.get('maxTrainingMoleculeCount', -1)) == maxTrainingMoleculeCount
        and int(cachedFeatureMetadata.get('sampleStride', -1)) == sampleStride
        and cachedFeatureMetadata.get('targetColumnName') == targetColumnName
    )

if cacheIsReusable:
    featureFrame = pd.read_csv(featureCsvPath)
    skippedMoleculeCount = int(cachedFeatureMetadata.get('skippedMoleculeCount', 0))
    elapsedSeconds = float(cachedFeatureMetadata.get('elapsedSeconds', 0.0))
    print('Memuat fitur dari cache yang cocok dengan konfigurasi notebook saat ini.')
else:
    startTime = time.perf_counter()
    featureRows = []
    skippedMoleculeCount = 0

    for sampledIndex, blockLines in enumerate(
        iterSdfBlocks(sdfPath, maxMoleculeCount=maxTrainingMoleculeCount, sampleStride=sampleStride),
        start=1,
    ):
        featureRow = extractMoleculeFeatures(blockLines, sampledIndex)
        if featureRow is None:
            skippedMoleculeCount += 1
            continue
        featureRows.append(featureRow)

        if sampledIndex % reportInterval == 0:
            print(f'Sudah memproses {sampledIndex:,} molekul untuk preprocessing...')

    featureFrame = pd.DataFrame(featureRows)
    elapsedSeconds = time.perf_counter() - startTime

    if featureFrame.empty:
        raise RuntimeError('Tidak ada fitur yang berhasil diekstrak. Periksa file SDF atau parser.')

    featureFrame.to_csv(featureCsvPath, index=False)
    cachedFeatureMetadata = {
        'sdfPath': str(sdfPath),
        'maxTrainingMoleculeCount': int(maxTrainingMoleculeCount),
        'sampleStride': int(sampleStride),
        'targetColumnName': targetColumnName,
        'skippedMoleculeCount': int(skippedMoleculeCount),
        'elapsedSeconds': float(elapsedSeconds),
    }
    with featureMetadataPath.open('w', encoding='utf-8') as handle:
        json.dump(cachedFeatureMetadata, handle, ensure_ascii=False, indent=2)

preprocessSummaryFrame = pd.DataFrame(
    {
        'informasi': [
            'jumlahBarisFitur',
            'jumlahKolomFitur',
            'nilaiHilangTotal',
            'skippedMoleculeCount',
            'waktuPreprocessDetik',
            'targetMinimum',
            'targetMaksimum',
            'targetRataRata',
        ],
        'nilai': [
            int(len(featureFrame)),
            int(featureFrame.shape[1]),
            int(featureFrame.isna().sum().sum()),
            int(skippedMoleculeCount),
            float(elapsedSeconds),
            float(featureFrame[targetColumnName].min()),
            float(featureFrame[targetColumnName].max()),
            float(featureFrame[targetColumnName].mean()),
        ],
    }
)

print('Ringkasan hasil preprocessing:')
display(preprocessSummaryFrame)
display(featureFrame.head())
print(f'Fitur tersimpan di: {featureCsvPath}')


## Pembagian Data

Bagian ini membagi data menjadi train, validasi, dan test agar model tabular maupun `ginGraphNetwork` dibandingkan pada protokol evaluasi yang sama.


In [ ]:
modelingFrame = featureFrame[modelFeatureColumns + [targetColumnName]].dropna().copy()
xData = modelingFrame[modelFeatureColumns]
yData = modelingFrame[targetColumnName]

xTrainPool, xTest, yTrainPool, yTest = train_test_split(
    xData,
    yData,
    test_size=localTestFraction,
    random_state=randomState,
)

validFractionWithinTrainPool = localValidFraction / (1.0 - localTestFraction)
xTrain, xValid, yTrain, yValid = train_test_split(
    xTrainPool,
    yTrainPool,
    test_size=validFractionWithinTrainPool,
    random_state=randomState,
)

splitFrame = pd.DataFrame(
    {
        'bagianData': ['train', 'validasi', 'test'],
        'jumlahBaris': [len(xTrain), len(xValid), len(xTest)],
        'fraksi': [len(xTrain) / len(modelingFrame), len(xValid) / len(modelingFrame), len(xTest) / len(modelingFrame)],
    }
)

print('Ringkasan pembagian data:')
display(splitFrame)


## Pelatihan Model

Bagian ini melatih seluruh kandidat model dari `main.py`, membandingkan validasi untuk model tabular dan `ginGraphNetwork`, lalu memilih model terbaik berdasarkan nilai MAE pada data validasi. Untuk `ginGraphNetwork`, training memakai early stopping dengan `patience=3` agar proses berhenti saat metrik validasi tidak lagi membaik.


In [ ]:
modelMap = build_model_map(randomState)

resultRows = []
graphModelResult = None

for modelName, modelObject in modelMap.items():
    startTime = time.perf_counter()
    modelObject.fit(xTrain, yTrain)
    validPrediction = modelObject.predict(xValid)
    durationSeconds = time.perf_counter() - startTime

    resultRows.append(
        {
            'modelName': modelName,
            'validMae': mean_absolute_error(yValid, validPrediction),
            'validRmse': math.sqrt(mean_squared_error(yValid, validPrediction)),
            'validR2': r2_score(yValid, validPrediction),
            'durasiDetik': durationSeconds,
        }
    )
    print(f'Model {modelName} selesai dilatih.')

graphTrainingMoleculeCount = maxTrainingMoleculeCount
print(f'Memulai training graph network pada {graphTrainingMoleculeCount:,} molekul.')
try:
    graphModelResult = train_gin_regressor(
        sdf_path=sdfPath,
        max_molecule_count=graphTrainingMoleculeCount,
        random_state=randomState,
        valid_fraction=localValidFraction,
        test_fraction=localTestFraction,
        batch_size=64,
        hidden_dim=96,
        layer_count=4,
        dropout=0.10,
        epoch_count=100,
        learning_rate=1e-3,
        patience=3,
        log_every=10_000,
        verbose=True,
    )
    resultRows.append(
        {
            'modelName': graphModelResult['modelName'],
            'validMae': graphModelResult['validMae'],
            'validRmse': graphModelResult['validRmse'],
            'validR2': graphModelResult['validR2'],
            'durasiDetik': graphModelResult['durasiDetik'],
        }
    )
    print(
        f"Model {graphModelResult['modelName']} selesai dilatih pada {graphModelResult['sampleCount']:,} graf."
    )
except Exception as graphError:
    print(f'Model ginGraphNetwork dilewati: {graphError}')

resultFrame = pd.DataFrame(resultRows).sort_values('validMae').reset_index(drop=True)
bestModelName = str(resultFrame.loc[0, 'modelName'])

print('Perbandingan model pada data validasi:')
display(resultFrame)
print(f'Model terbaik sementara: {bestModelName}')


## Evaluasi Model

Bagian ini menguji model terbaik pada data test. Jika model terbaik adalah `ginGraphNetwork`, hasil evaluasi diambil dari training graf; jika bukan, model tabular dilatih ulang pada gabungan train dan validasi sebelum diuji pada test. Informasi early stopping seperti epoch terbaik dan epoch berhenti juga ikut dibawa ke ringkasan hasil saat model graph dipakai.


In [ ]:
graphHistoryFrame = pd.DataFrame(columns=['epoch', 'trainLoss', 'validMae', 'validRmse'])

if bestModelName == 'ginGraphNetwork':
    if graphModelResult is None:
        raise RuntimeError('Hasil GIN tidak tersedia meskipun terpilih sebagai model terbaik.')

    bestModel = graphModelResult['modelObject']
    testMae = float(graphModelResult['testMae'])
    testRmse = float(graphModelResult['testRmse'])
    testR2 = float(graphModelResult['testR2'])
    predictionFrame = graphModelResult['predictionFrame'].copy()
    graphHistoryFrame = pd.DataFrame(graphModelResult.get('historyRows', []))
    evaluationDataSource = 'graphNetwork'
else:
    bestModel = clone(modelMap[bestModelName])
    xTrainCombined = pd.concat([xTrain, xValid], axis=0)
    yTrainCombined = pd.concat([yTrain, yValid], axis=0)

    bestModel.fit(xTrainCombined, yTrainCombined)
    testPrediction = bestModel.predict(xTest)

    testMae = mean_absolute_error(yTest, testPrediction)
    testRmse = math.sqrt(mean_squared_error(yTest, testPrediction))
    testR2 = r2_score(yTest, testPrediction)

    predictionFrame = pd.DataFrame(
        {
            'nilaiAktual': yTest.to_numpy(),
            'nilaiPrediksi': testPrediction,
        }
    )
    predictionFrame['galatAbsolut'] = (predictionFrame['nilaiAktual'] - predictionFrame['nilaiPrediksi']).abs()
    if graphModelResult is not None:
        graphHistoryFrame = pd.DataFrame(graphModelResult.get('historyRows', []))
    evaluationDataSource = 'fiturTabular'

predictionFrame['residual'] = predictionFrame['nilaiPrediksi'] - predictionFrame['nilaiAktual']
predictionFrame['galatKuadrat'] = predictionFrame['residual'] ** 2
predictionFrame['galatRelatifAbsolut'] = predictionFrame['galatAbsolut'] / np.maximum(predictionFrame['nilaiAktual'].abs(), 1e-8)

testMse = mean_squared_error(predictionFrame['nilaiAktual'], predictionFrame['nilaiPrediksi'])
testMedAe = median_absolute_error(predictionFrame['nilaiAktual'], predictionFrame['nilaiPrediksi'])
testExplainedVariance = explained_variance_score(predictionFrame['nilaiAktual'], predictionFrame['nilaiPrediksi'])
testMape = float(predictionFrame['galatRelatifAbsolut'].mean())
testSmape = float(
    np.mean(
        2.0
        * predictionFrame['galatAbsolut']
        / np.maximum(predictionFrame['nilaiAktual'].abs() + predictionFrame['nilaiPrediksi'].abs(), 1e-8)
    )
)
pearsonCorrelation = float(
    np.corrcoef(predictionFrame['nilaiAktual'], predictionFrame['nilaiPrediksi'])[0, 1]
) if len(predictionFrame) > 1 else np.nan

metricFrame = pd.DataFrame(
    {
        'metrik': [
            'MAE',
            'MSE',
            'RMSE',
            'MedianAE',
            'MAPE',
            'sMAPE',
            'R2',
            'ExplainedVariance',
            'PearsonCorrelation',
            'jumlahDataTest',
        ],
        'nilai': [
            float(testMae),
            float(testMse),
            float(testRmse),
            float(testMedAe),
            float(testMape),
            float(testSmape),
            float(testR2),
            float(testExplainedVariance),
            float(pearsonCorrelation),
            float(len(predictionFrame)),
        ],
    }
)

residualSummaryFrame = pd.DataFrame(
    {
        'ringkasan': ['meanResidual', 'stdResidual', 'minResidual', 'maxResidual'],
        'nilai': [
            float(predictionFrame['residual'].mean()),
            float(predictionFrame['residual'].std()),
            float(predictionFrame['residual'].min()),
            float(predictionFrame['residual'].max()),
        ],
    }
)

print('Hasil evaluasi pada data test:')
display(metricFrame)
print('Contoh prediksi pada data test:')
display(predictionFrame.head(10))
print('Ringkasan residual:')
display(residualSummaryFrame)
if not graphHistoryFrame.empty:
    print('Ringkasan training graph network:')
    display(graphHistoryFrame.tail())


## Visualisasi Hasil

Bagian ini menampilkan plot validasi model, distribusi target, aktual vs prediksi, residual, korelasi fitur, dan riwayat training GIN bila model graph ikut dijalankan.


In [ ]:
figureDir = outputDir / 'figures'
figureDir.mkdir(parents=True, exist_ok=True)
plotPathMap = {}

correlationSeries = (
    featureFrame[modelFeatureColumns + [targetColumnName]]
    .corr(numeric_only=True)[targetColumnName]
    .drop(targetColumnName)
    .abs()
    .sort_values(ascending=False)
    .head(12)
)
correlationFrame = correlationSeries.rename('korelasiAbsolut').reset_index()
correlationFrame.columns = ['featureName', 'korelasiAbsolut']

modelComparisonFigurePath = figureDir / 'perbandingan-model-validasi.png'
evaluationFigurePath = figureDir / 'evaluasi-model-terbaik.png'
regressionDiagnosticFigurePath = figureDir / 'diagnostik-regresi.png'
featureSignalFigurePath = figureDir / 'sinyal-fitur.png'
graphHistoryFigurePath = figureDir / 'riwayat-training-gin.png'

plt.figure(figsize=(10, 5.5))
sns.barplot(data=resultFrame.sort_values('validMae'), x='validMae', y='modelName', color='#0f766e')
plt.title('Perbandingan Model pada Data Validasi')
plt.xlabel('MAE Validasi')
plt.ylabel('Model')
plt.tight_layout()
plt.savefig(modelComparisonFigurePath, dpi=180, bbox_inches='tight')
plotPathMap['modelComparison'] = str(modelComparisonFigurePath)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(featureFrame[targetColumnName], bins=40, kde=True, ax=axes[0, 0], color='#0f766e')
axes[0, 0].set_title('Distribusi Target Radius Gyration')
axes[0, 0].set_xlabel('radiusGyration')
axes[0, 0].set_ylabel('Frekuensi')

axes[0, 1].scatter(predictionFrame['nilaiAktual'], predictionFrame['nilaiPrediksi'], alpha=0.35, color='#0f172a')
minimumValue = min(predictionFrame['nilaiAktual'].min(), predictionFrame['nilaiPrediksi'].min())
maximumValue = max(predictionFrame['nilaiAktual'].max(), predictionFrame['nilaiPrediksi'].max())
axes[0, 1].plot([minimumValue, maximumValue], [minimumValue, maximumValue], linestyle='--', color='black')
axes[0, 1].set_title('Aktual vs Prediksi')
axes[0, 1].set_xlabel('Nilai Aktual')
axes[0, 1].set_ylabel('Nilai Prediksi')

sns.histplot(predictionFrame['galatAbsolut'], bins=35, kde=True, ax=axes[1, 0], color='#1d4ed8')
axes[1, 0].set_title('Distribusi Galat Absolut')
axes[1, 0].set_xlabel('galatAbsolut')
axes[1, 0].set_ylabel('Frekuensi')

axes[1, 1].scatter(predictionFrame['nilaiPrediksi'], predictionFrame['residual'], alpha=0.35, color='#7c3aed')
axes[1, 1].axhline(0.0, linestyle='--', color='black')
axes[1, 1].set_title('Residual vs Prediksi')
axes[1, 1].set_xlabel('Nilai Prediksi')
axes[1, 1].set_ylabel('Residual')

plt.tight_layout()
plt.savefig(evaluationFigurePath, dpi=180, bbox_inches='tight')
plotPathMap['evaluation'] = str(evaluationFigurePath)
plt.show()

sortedPredictionFrame = predictionFrame.sort_values('nilaiAktual').reset_index(drop=True)
quantileGrid = np.linspace(0.05, 0.95, 19)
quantileFrame = pd.DataFrame(
    {
        'kuantil': quantileGrid,
        'aktual': np.quantile(predictionFrame['nilaiAktual'], quantileGrid),
        'prediksi': np.quantile(predictionFrame['nilaiPrediksi'], quantileGrid),
    }
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(sortedPredictionFrame.index, sortedPredictionFrame['nilaiAktual'], label='Aktual', color='#0f172a')
axes[0].plot(sortedPredictionFrame.index, sortedPredictionFrame['nilaiPrediksi'], label='Prediksi', color='#0f766e', alpha=0.85)
axes[0].set_title('Perbandingan Aktual dan Prediksi Terurut')
axes[0].set_xlabel('Urutan Sampel Test')
axes[0].set_ylabel('Nilai Target')
axes[0].legend()

axes[1].plot(quantileFrame['kuantil'], quantileFrame['aktual'], marker='o', label='Aktual', color='#0f172a')
axes[1].plot(quantileFrame['kuantil'], quantileFrame['prediksi'], marker='o', label='Prediksi', color='#1d4ed8')
axes[1].set_title('Perbandingan Kuantil Aktual vs Prediksi')
axes[1].set_xlabel('Kuantil')
axes[1].set_ylabel('Nilai Target')
axes[1].legend()

plt.tight_layout()
plt.savefig(regressionDiagnosticFigurePath, dpi=180, bbox_inches='tight')
plotPathMap['regressionDiagnostic'] = str(regressionDiagnosticFigurePath)
plt.show()

print('Fitur dengan korelasi absolut tertinggi terhadap target:')
display(correlationFrame)

def buildImportanceFrame(trainedModel, featureNames):
    if hasattr(trainedModel, 'feature_importances_'):
        importanceSeries = pd.Series(trainedModel.feature_importances_, index=featureNames)
    elif hasattr(trainedModel, 'named_steps') and 'ridge' in trainedModel.named_steps:
        importanceSeries = pd.Series(np.abs(trainedModel.named_steps['ridge'].coef_), index=featureNames)
    else:
        return None
    importanceFrame = importanceSeries.sort_values(ascending=False).rename('importance').reset_index()
    importanceFrame.columns = ['featureName', 'importance']
    return importanceFrame

importanceFrame = buildImportanceFrame(bestModel, modelFeatureColumns)
importanceExportFrame = pd.DataFrame(columns=['featureName', 'importance'])
featureSignalFrame = correlationFrame.rename(columns={'korelasiAbsolut': 'score'}).copy()
featureSignalTitle = '12 Fitur dengan Korelasi Tertinggi'
featureSignalLabel = 'Korelasi Absolut'

if importanceFrame is None:
    print('Model terbaik tidak menyediakan importance, sehingga plot fitur memakai korelasi absolut.')
else:
    importanceExportFrame = importanceFrame.copy()
    featureSignalFrame = importanceFrame.head(12).rename(columns={'importance': 'score'}).copy()
    featureSignalTitle = '12 Fitur Terpenting pada Model Terbaik'
    featureSignalLabel = 'Besaran Importance'
    display(importanceFrame.head(15))

plt.figure(figsize=(9, 6))
sns.barplot(data=featureSignalFrame.head(12), x='score', y='featureName', color='#0f766e')
plt.title(featureSignalTitle)
plt.xlabel(featureSignalLabel)
plt.ylabel('Nama Fitur')
plt.tight_layout()
plt.savefig(featureSignalFigurePath, dpi=180, bbox_inches='tight')
plotPathMap['featureSignal'] = str(featureSignalFigurePath)
plt.show()

if not graphHistoryFrame.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    axes[0].plot(graphHistoryFrame['epoch'], graphHistoryFrame['trainLoss'], marker='o', color='#0f766e')
    axes[0].set_title('Train Loss GIN per Epoch')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Smooth L1 Loss')

    axes[1].plot(graphHistoryFrame['epoch'], graphHistoryFrame['validMae'], marker='o', color='#1d4ed8', label='Valid MAE')
    axes[1].plot(graphHistoryFrame['epoch'], graphHistoryFrame['validRmse'], marker='o', color='#dc2626', label='Valid RMSE')
    axes[1].set_title('Validasi GIN per Epoch')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Nilai Metrik')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(graphHistoryFigurePath, dpi=180, bbox_inches='tight')
    plotPathMap['ginTrainingHistory'] = str(graphHistoryFigurePath)
    plt.show()


## Penyimpanan Hasil

Bagian ini menyimpan model terbaik, CSV evaluasi, plot, payload website, dan manifest aktif agar hasil run terbaru langsung bisa dibaca oleh web presentasi.


In [ ]:
artifactPath = outputDir / 'modelRegresiRadiusGyration.pkl'
predictionPath = outputDir / 'prediksiDataTest.csv'
metadataPath = outputDir / 'ringkasanEksperimen.json'
modelComparisonPath = outputDir / 'perbandinganModelValidasi.csv'
metricPath = outputDir / 'metrikEvaluasiModelTerbaik.csv'
correlationPath = outputDir / 'korelasiFitur.csv'
importancePath = outputDir / 'fiturTerpenting.csv'
graphHistoryPath = outputDir / 'riwayatTrainingGin.csv'
residualSummaryPath = outputDir / 'ringkasanResidual.csv'
quantileDiagnosticPath = outputDir / 'diagnostikKuantilRegresi.csv'
plotManifestPath = outputDir / 'daftarPlotEvaluasi.json'
presentationSummaryPath = outputDir / 'ringkasanPresentasi.json'

metricExportFrame = metricFrame.copy()
metricExportFrame['bestModelName'] = bestModelName
metricExportFrame['evaluationDataSource'] = evaluationDataSource

artifactPayload = {
    'bestModelName': bestModelName,
    'targetColumnName': targetColumnName,
    'featureColumnNames': modelFeatureColumns,
    'modelObject': bestModel,
    'testMetrics': {
        'mae': float(testMae),
        'mse': float(testMse),
        'rmse': float(testRmse),
        'medianAe': float(testMedAe),
        'mape': float(testMape),
        'smape': float(testSmape),
        'r2': float(testR2),
        'explainedVariance': float(testExplainedVariance),
        'pearsonCorrelation': float(pearsonCorrelation),
    },
    'evaluationDataSource': evaluationDataSource,
    'workflowType': 'pcqm4mv2-sdf-baseline',
    'plotPaths': plotPathMap,
}

with artifactPath.open('wb') as handle:
    pickle.dump(artifactPayload, handle)

predictionFrame.to_csv(predictionPath, index=False)
resultFrame.to_csv(modelComparisonPath, index=False)
metricExportFrame.to_csv(metricPath, index=False)
correlationFrame.to_csv(correlationPath, index=False)
importanceExportFrame.to_csv(importancePath, index=False)
graphHistoryFrame.to_csv(graphHistoryPath, index=False)
residualSummaryFrame.to_csv(residualSummaryPath, index=False)
quantileFrame.to_csv(quantileDiagnosticPath, index=False)

with plotManifestPath.open('w', encoding='utf-8') as handle:
    json.dump(plotPathMap, handle, ensure_ascii=False, indent=2)

presentationSummaryPayload = {
    'judul': 'Ringkasan Evaluasi Baseline PCQM4Mv2',
    'bestModelName': bestModelName,
    'evaluationDataSource': evaluationDataSource,
    'targetColumnName': targetColumnName,
    'jumlahData': int(len(featureFrame)),
    'jumlahDataTest': int(len(predictionFrame)),
    'topModelsByValidMae': json.loads(resultFrame.head(5).round(6).to_json(orient='records')),
    'testMetrics': {
        'mae': float(testMae),
        'mse': float(testMse),
        'rmse': float(testRmse),
        'medianAe': float(testMedAe),
        'mape': float(testMape),
        'smape': float(testSmape),
        'r2': float(testR2),
        'explainedVariance': float(testExplainedVariance),
        'pearsonCorrelation': float(pearsonCorrelation),
    },
    'featureSignalPreview': json.loads(featureSignalFrame.head(12).round(6).to_json(orient='records')),
    'savedTables': {
        'modelComparisonPath': str(modelComparisonPath),
        'metricPath': str(metricPath),
        'predictionPath': str(predictionPath),
        'correlationPath': str(correlationPath),
        'importancePath': str(importancePath),
        'graphHistoryPath': str(graphHistoryPath),
        'residualSummaryPath': str(residualSummaryPath),
        'quantileDiagnosticPath': str(quantileDiagnosticPath),
    },
    'savedPlots': plotPathMap,
}
if graphModelResult is not None:
    presentationSummaryPayload['graphTraining'] = {
        'sampleCount': int(graphModelResult['sampleCount']),
        'trainedEpochCount': int(graphModelResult['trainedEpochCount']),
        'bestEpoch': int(graphModelResult['bestEpoch']),
        'stopEpoch': int(graphModelResult['stopEpoch']),
        'maxEpochCount': int(graphModelResult['maxEpochCount']),
        'earlyStoppingPatience': int(graphModelResult['earlyStoppingPatience']),
        'stoppedEarly': bool(graphModelResult['stoppedEarly']),
        'device': graphModelResult['device'],
        'validMae': float(graphModelResult['validMae']),
        'testMae': float(graphModelResult['testMae']),
    }

with presentationSummaryPath.open('w', encoding='utf-8') as handle:
    json.dump(presentationSummaryPayload, handle, ensure_ascii=False, indent=2)

metadataPayload = {
    'sdfPath': str(sdfPath),
    'jumlahData': int(len(featureFrame)),
    'bestModelName': bestModelName,
    'targetColumnName': targetColumnName,
    'evaluationDataSource': evaluationDataSource,
    'evaluationProtocol': 'holdout split pada data training SDF',
    'officialReferences': [
        'https://ogb.stanford.edu/docs/lsc/pcqm4mv2/',
        'https://ogb.stanford.edu/docs/lsc/leaderboards/#pcqm4mv2',
    ],
    'testMetrics': {
        'mae': float(testMae),
        'mse': float(testMse),
        'rmse': float(testRmse),
        'medianAe': float(testMedAe),
        'mape': float(testMape),
        'smape': float(testSmape),
        'r2': float(testR2),
        'explainedVariance': float(testExplainedVariance),
        'pearsonCorrelation': float(pearsonCorrelation),
    },
    'savedTables': {
        'modelComparisonPath': str(modelComparisonPath),
        'metricPath': str(metricPath),
        'predictionPath': str(predictionPath),
        'correlationPath': str(correlationPath),
        'importancePath': str(importancePath),
        'graphHistoryPath': str(graphHistoryPath),
        'residualSummaryPath': str(residualSummaryPath),
        'quantileDiagnosticPath': str(quantileDiagnosticPath),
        'presentationSummaryPath': str(presentationSummaryPath),
    },
    'savedPlots': plotPathMap,
}
if graphModelResult is not None:
    metadataPayload['graphTraining'] = {
        'sampleCount': int(graphModelResult['sampleCount']),
        'trainedEpochCount': int(graphModelResult['trainedEpochCount']),
        'bestEpoch': int(graphModelResult['bestEpoch']),
        'stopEpoch': int(graphModelResult['stopEpoch']),
        'maxEpochCount': int(graphModelResult['maxEpochCount']),
        'earlyStoppingPatience': int(graphModelResult['earlyStoppingPatience']),
        'stoppedEarly': bool(graphModelResult['stoppedEarly']),
        'device': graphModelResult['device'],
    }

with metadataPath.open('w', encoding='utf-8') as handle:
    json.dump(metadataPayload, handle, ensure_ascii=False, indent=2)

webAssetResult = sync_notebook_web_assets(
    sdf_path=sdfPath,
    artifact_dir=outputDir,
    notebook_path=notebookPath,
    notebook_config={
        'maxMoleculeCount': maxTrainingMoleculeCount,
        'sampleStride': sampleStride,
        'reportInterval': reportInterval,
        'targetColumnName': targetColumnName,
        'featureCount': len(modelFeatureColumns),
        'featureColumnNames': modelFeatureColumns,
        'randomState': randomState,
        'graphEpochCount': 100,
        'graphEarlyStoppingPatience': 3,
    },
    viewer_config=viewerExportConfig,
    showcase_config=showcaseExportConfig,
)

print(f'Model tersimpan di: {artifactPath}')
print(f'Prediksi test tersimpan di: {predictionPath}')
print(f'Perbandingan model tersimpan di: {modelComparisonPath}')
print(f'Metrik evaluasi tersimpan di: {metricPath}')
print(f'Ringkasan residual tersimpan di: {residualSummaryPath}')
print(f'Diagnostik kuantil regresi tersimpan di: {quantileDiagnosticPath}')
print(f'Ringkasan presentasi tersimpan di: {presentationSummaryPath}')
print(f'Ringkasan eksperimen tersimpan di: {metadataPath}')
print(f"Showcase website tersimpan di: {webAssetResult['showcasePath']}")
if 'viewerManifestPath' in webAssetResult:
    print(f"Data viewer 3D tersimpan di: {webAssetResult['viewerManifestPath']}")
print(f"Web manifest aktif tersimpan di: {webAssetResult['webManifestPath']}")
